# llm-abstention-tuning — full run on Colab

Post-trains `Qwen2.5-1.5B-Instruct` (LoRA SFT → LoRA DPO) to abstain instead of hallucinating, then evaluates on
TriviaQA (in-distribution), PopQA and NQ-Open (out-of-distribution).

**Runtime → Change runtime type → GPU.** L4 or A100: use the main config. Free T4: use the quick config.
Each stage is its own cell and writes to Google Drive, so if Colab disconnects, re-run the setup cells and continue
from the stage that was interrupted.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Setup

In [ ]:
REPO = "https://github.com/manojrevenkar12-web/llm-abstention-tuning.git"
!git clone -q $REPO /content/llm-abstention-tuning 2>/dev/null || git -C /content/llm-abstention-tuning pull -q
%cd /content/llm-abstention-tuning
!pip install -q -r requirements.txt && pip install -q -e .

In [ ]:
# Keep run outputs on Google Drive so a disconnect does not lose finished stages.
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE_RUNS = "/content/drive/MyDrive/llm-abstention-tuning/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)
if not os.path.islink("runs"):
    !rm -rf runs && ln -s $DRIVE_RUNS runs
!ls -la runs

In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
CONFIG = "configs/qwen2.5-1.5b-quick.yaml" if "T4" in gpu else "configs/qwen2.5-1.5b.yaml"
print(gpu, "->", CONFIG)

## 2. Sanity check — tests and tiny end-to-end run (~1 min)

In [ ]:
!pytest -q

## 3. Pipeline

In [ ]:
# Download and split the datasets (train questions overlapping any eval set are dropped)
!python -m abstention_tuning prepare --config $CONFIG

In [ ]:
# Probe what the base model knows: greedy + K samples per question. Longest stage.
!python -m abstention_tuning probe --config $CONFIG

In [ ]:
# Build SFT and DPO data from the probe labels
!python -m abstention_tuning build --config $CONFIG
import json, glob
print(json.dumps(json.load(open(glob.glob("runs/*/datasets/build_stats.json")[-1])), indent=2))

In [ ]:
!python -m abstention_tuning sft --config $CONFIG

In [ ]:
!python -m abstention_tuning dpo --config $CONFIG

In [ ]:
# Evaluate base, base_prompted, sft and sft_dpo on every eval set
!python -m abstention_tuning evaluate --config $CONFIG

In [ ]:
# Release gates (non-zero exit = a blocking gate failed; the report still records it), then the report
!python -m abstention_tuning gates --config $CONFIG; python -m abstention_tuning report --config $CONFIG

## 4. Results

In [ ]:
import yaml
from IPython.display import Markdown, Image, display
run_name = yaml.safe_load(open(CONFIG))["run_name"]
REPORT_DIR = f"runs/{run_name}/report"
display(Markdown(open(f"{REPORT_DIR}/REPORT.md").read().replace("](", f"]({REPORT_DIR}/")))
for img in ["outcomes", "risk_coverage", "reliability", "training"]:
    p = f"{REPORT_DIR}/{img}.png"
    if os.path.exists(p):
        display(Image(p))

## 5. Save results for the repo

Copies the report, figures and metrics (not the models) into `results/<run_name>/` and zips them for download.
Commit that folder to GitHub, then fill the README results table and `MODEL_CARD.md` from `REPORT.md`.

In [ ]:
import shutil
dst = f"results/{run_name}"
os.makedirs(dst, exist_ok=True)
for f in glob.glob(f"{REPORT_DIR}/*"):
    shutil.copy(f, dst)
shutil.copy(f"runs/{run_name}/eval/metrics.json", dst)
shutil.copy(f"runs/{run_name}/datasets/build_stats.json", dst)
shutil.copy(f"runs/{run_name}/config.resolved.yaml", dst)
shutil.make_archive(f"/content/results_{run_name}", "zip", dst)
from google.colab import files
files.download(f"/content/results_{run_name}.zip")

## 6. (Optional) Publish the adapters to the Hugging Face Hub

Needs a Hub write token (`huggingface-cli login` or the Colab secret `HF_TOKEN`).

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# repo_id = "<your-hf-username>/qwen2.5-1.5b-abstention-lora"
# api.create_repo(repo_id, exist_ok=True)
# api.upload_folder(folder_path=f"runs/{run_name}/models/dpo_adapter", repo_id=repo_id, path_in_repo="dpo_adapter")
# api.upload_folder(folder_path=f"runs/{run_name}/models/sft_adapter", repo_id=repo_id, path_in_repo="sft_adapter")